In [1]:
"""
Analysis Module for Animal Classification with Continuous Attribute Learning (CAL)
This module provides tools for analyzing and visualizing the performance of the CAL model.
It includes functions for generating confusion matrices, class-wise accuracy metrics,
and attribute prediction analysis.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import pandas as pd
import numpy as np
from PIL import Image
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Enable GPU optimizations for faster analysis
torch.backends.cudnn.benchmark = True

# Configuration parameters for the analysis
CONFIG = {
    'data_path': '/kaggle/input/vlg-recruitment-24-challenge/vlg-dataset/train',
    'model_path': '/kaggle/input/pixelplay2/best_model.pth',  # Path to trained model
    'batch_size': 32,          # Batch size for analysis
    'num_attributes': 85,      # Number of continuous attributes
    'num_classes': 50,         # Number of animal classes
    'image_size': 224         # Input image size
}

class AnimalAttributeDataset(Dataset):
    """
    Custom Dataset class for loading animal images with their attributes and labels.
    Used during analysis phase to load and preprocess images consistently.
    
    Args:
        image_paths (list): List of paths to image files
        labels (list): List of class labels corresponding to images
        attributes (np.array): Matrix of continuous attributes for each class
        transform (callable, optional): Optional transform to be applied to images
    """
    def __init__(self, image_paths, labels, attributes, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.attributes = attributes
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform:
                image = self.transform(image)
            label = self.labels[idx]
            attribute = self.attributes[label]
            return image, torch.tensor(label, dtype=torch.long), torch.tensor(attribute, dtype=torch.float32)
        except Exception as e:
            print(f"Error loading image {self.image_paths[idx]}: {str(e)}")
            return None

def collate_fn(batch):
    """
    Custom collate function to handle None values in batch.
    Filters out None values that might occur due to corrupted images.
    """
    batch = list(filter(lambda x: x is not None, batch))
    if len(batch) == 0:
        raise RuntimeError("Batch is empty after filtering")
    return torch.utils.data.dataloader.default_collate(batch)

class CALModel(nn.Module):
    """
    Model architecture for analysis.
    Must match the architecture used during training.
    """
    def __init__(self, num_attributes, num_classes):
        super(CALModel, self).__init__()
        # Load pretrained ResNet backbone
        self.backbone = models.resnet50(pretrained=True)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        # Attribute prediction head
        self.attribute_predictor = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.BatchNorm1d(512),
            nn.Linear(512, num_attributes),
            nn.Sigmoid()
        )
        
        # Class prediction head
        self.class_predictor = nn.Sequential(
            nn.Linear(num_attributes, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.BatchNorm1d(512),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        attributes = self.attribute_predictor(features)
        classes = self.class_predictor(attributes)
        return attributes, classes

def load_data():
    """
    Loads and preprocesses the dataset for analysis.
    
    Returns:
        tuple: (class_mapping, continuous_attributes, predicate_names)
            - class_mapping: Dictionary mapping class names to indices
            - continuous_attributes: Normalized attribute matrix
            - predicate_names: List of attribute names
    """
    # Load class mapping
    class_df = pd.read_csv('/kaggle/input/vlg-recruitment-24-challenge/vlg-dataset/classes.txt', 
                          delimiter='\s+', header=None, names=['index', 'class_name'])
    class_mapping = {row['class_name']: row['index']-1 for _, row in class_df.iterrows()}
    
    # Load and normalize continuous attributes
    continuous_attributes = pd.read_csv('/kaggle/input/vlg-recruitment-24-challenge/vlg-dataset/predicate-matrix-continuous.txt', 
                                      delimiter='\s+', header=None).values.astype(np.float32)
    continuous_attributes = (continuous_attributes - continuous_attributes.min()) / (continuous_attributes.max() - continuous_attributes.min())
    
    # Load predicate (attribute) names
    predicate_names = pd.read_csv('/kaggle/input/vlg-recruitment-24-challenge/vlg-dataset/predicates.txt', 
                                 delimiter='\s+', header=None, names=['index', 'predicate'])['predicate'].values
    
    return class_mapping, continuous_attributes, predicate_names

def prepare_data(class_mapping, continuous_attributes):
    """
    Prepares data loaders for analysis.
    Uses a consistent transform pipeline without augmentation for reliable analysis.
    
    Args:
        class_mapping: Dictionary mapping class names to indices
        continuous_attributes: Matrix of continuous attributes
        
    Returns:
        tuple: (train_loader, val_loader) for analysis
    """
    image_paths = []
    labels = []
    
    # Collect all image paths and labels
    for class_name in os.listdir(CONFIG['data_path']):
        if class_name in class_mapping:
            class_path = os.path.join(CONFIG['data_path'], class_name)
            if os.path.isdir(class_path):
                class_label = class_mapping[class_name]
                for img_name in os.listdir(class_path):
                    if img_name.endswith(('.jpg', '.jpeg', '.png')):
                        img_path = os.path.join(class_path, img_name)
                        image_paths.append(img_path)
                        labels.append(class_label)
    
    # Split data into training and validation sets
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        image_paths, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    # Define consistent transform pipeline for analysis
    transform = transforms.Compose([
        transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Create datasets and dataloaders
    train_dataset = AnimalAttributeDataset(train_paths, train_labels, continuous_attributes, transform)
    val_dataset = AnimalAttributeDataset(val_paths, val_labels, continuous_attributes, transform)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,  # No shuffling for consistent analysis
        num_workers=2,
        pin_memory=True,
        collate_fn=collate_fn
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=collate_fn
    )
    
    return train_loader, val_loader

def analyze_model_performance(model, data_loader, split_name, device):
    """
    Analyzes model performance on a given dataset.
    
    Args:
        model: The trained CAL model
        data_loader: DataLoader containing the dataset to analyze
        split_name: Name of the dataset split (e.g., 'Training' or 'Validation')
        device: Device to run the analysis on
        
    Returns:
        dict: Dictionary containing predictions and ground truth for both classes and attributes
    """
    model.eval()
    predictions = []
    labels = []
    attr_predictions = []
    attr_true = []
    
    print(f"Analyzing {split_name} data...")
    with torch.no_grad():
        for images, batch_labels, attributes in tqdm(data_loader):
            images = images.to(device)
            pred_attributes, pred_classes = model(images)
            _, predicted = torch.max(pred_classes, 1)
            
            predictions.extend(predicted.cpu().numpy())
            labels.extend(batch_labels.numpy())
            attr_predictions.append(pred_attributes.cpu().numpy())
            attr_true.append(attributes.numpy())
    
    attr_predictions = np.concatenate(attr_predictions)
    attr_true = np.concatenate(attr_true)
    
    return {
        'predictions': predictions,
        'labels': labels,
        'attr_predictions': attr_predictions,
        'attr_true': attr_true
    }

def plot_metrics(results, class_mapping, predicate_names, split_name):
    """
    Generates and saves visualization plots for model analysis.
    
    Args:
        results: Dictionary containing model predictions and ground truth
        class_mapping: Dictionary mapping class names to indices
        predicate_names: List of attribute names
        split_name: Name of the dataset split
        
    Returns:
        dict: Dictionary containing computed metrics
    """
    # Create output directory for plots
    os.makedirs('/kaggle/working/analysis', exist_ok=True)
    
    # Generate and save confusion matrix
    plt.figure(figsize=(20, 20))
    cm = confusion_matrix(results['labels'], results['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{split_name} Confusion Matrix')
    plt.savefig(f'/kaggle/working/analysis/{split_name.lower()}_confusion_matrix.png')
    plt.close()
    
    # Generate and save class-wise accuracy plot
    plt.figure(figsize=(15, 5))
    report = classification_report(results['labels'], results['predictions'], output_dict=True)
    class_accuracies = {k: v['f1-score'] for k, v in report.items() if k.isdigit()}
    rev_class_mapping = {v: k for k, v in class_mapping.items()}
    class_accuracies = {rev_class_mapping[int(k)]: v for k, v in class_accuracies.items()}
    
    pd.Series(class_accuracies).sort_values().plot(kind='bar')
    plt.title(f'{split_name} Class-wise F1 Scores')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/analysis/{split_name.lower()}_class_accuracies.png')
    plt.close()
    
    # Generate and save attribute prediction accuracy plot
    attr_acc = np.mean((results['attr_predictions'] > 0.5) == results['attr_true'], axis=0)
    plt.figure(figsize=(15, 5))
    pd.Series(attr_acc, index=predicate_names).sort_values().plot(kind='bar')
    plt.title(f'{split_name} Attribute Prediction Accuracy')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/analysis/{split_name.lower()}_attribute_accuracies.png')
    plt.close()
    
    return {
        'overall_accuracy': report['accuracy'],
        'class_accuracies': class_accuracies,
        'attribute_accuracies': dict(zip(predicate_names, attr_acc))
    }

def print_analysis_summary(metrics, split_name):
    """
    Prints a summary of the analysis results.
    
    Args:
        metrics: Dictionary containing computed metrics
        split_name: Name of the dataset split
    """
    print(f"\n{split_name} Performance Summary:")
    print(f"Overall Accuracy: {metrics['overall_accuracy']:.4f}")
    
    # Print top performing classes
    print("\nTop 5 Best Performing Classes:")
    top_classes = pd.Series(metrics['class_accuracies']).nlargest(5)
    for class_name, acc in top_classes.items():
        print(f"{class_name}: {acc:.4f}")
    
    # Print worst performing classes
    print("\nBottom 5 Performing Classes:")
    bottom_classes = pd.Series(metrics['class_accuracies']).nsmallest(5)
    for class_name, acc in bottom_classes.items():
        print(f"{class_name}: {acc:.4f}")
    
    # Print best predicted attributes
    print("\nTop 5 Best Predicted Attributes:")
    top_attrs = pd.Series(metrics['attribute_accuracies']).nlargest(5)
    for attr, acc in top_attrs.items():
        print(f"{attr}: {acc:.4f}")
    
    # Print worst predicted attributes
    print("\nBottom 5 Predicted Attributes:")
    bottom_attrs = pd.Series(metrics['attribute_accuracies']).nsmallest(5)
    for attr, acc in bottom_attrs.items():
        print(f"{attr}: {acc:.4f}")

def main():
    """
    Main function to run the analysis pipeline.
    """
    # Setup device for analysis
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load necessary data
    class_mapping, continuous_attributes, predicate_names = load_data()
    train_loader, val_loader = prepare_data(class_mapping, continuous_attributes)
    
    # Load trained model
    model = CALModel(CONFIG['num_attributes'], CONFIG['num_classes'])
    checkpoint = torch.load(CONFIG['model_path'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    
    # Run analysis on both training and validation sets
    splits = [('Training', train_loader), ('Validation', val_loader)]
    
    for split_name, loader in splits:
        results = analyze_model_performance(model, loader, split_name, device)
        metrics = plot_metrics(results, class_mapping, predicate_names, split_name)
        print_analysis_summary(metrics, split_name)

In [2]:
if __name__ == "__main__":
    main()

Using device: cuda


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 205MB/s]
<ipython-input-1-78f69c1a7e16>:254: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is poss

Analyzing Training data...


100%|██████████| 239/239 [01:33<00:00,  2.55it/s]



Training Performance Summary:
Overall Accuracy: 1.0000

Top 5 Best Performing Classes:
antelope: 1.0000
grizzly+bear: 1.0000
killer+whale: 1.0000
beaver: 1.0000
dalmatian: 1.0000

Bottom 5 Performing Classes:
antelope: 1.0000
grizzly+bear: 1.0000
killer+whale: 1.0000
beaver: 1.0000
dalmatian: 1.0000

Top 5 Best Predicted Attributes:
black: 0.0262
white: 0.0262
blue: 0.0262
brown: 0.0262
spots: 0.0262

Bottom 5 Predicted Attributes:
gray: 0.0000
orange: 0.0000
red: 0.0000
yellow: 0.0000
patches: 0.0000
Analyzing Validation data...


100%|██████████| 60/60 [00:23<00:00,  2.53it/s]



Validation Performance Summary:
Overall Accuracy: 0.9036

Top 5 Best Performing Classes:
tiger: 1.0000
giraffe: 1.0000
zebra: 1.0000
giant+panda: 0.9901
leopard: 0.9899

Bottom 5 Performing Classes:
humpback+whale: 0.6531
blue+whale: 0.6765
mouse: 0.7123
rat: 0.7706
beaver: 0.8267

Top 5 Best Predicted Attributes:
black: 0.0262
white: 0.0262
blue: 0.0262
spots: 0.0257
brown: 0.0230

Bottom 5 Predicted Attributes:
gray: 0.0000
orange: 0.0000
red: 0.0000
yellow: 0.0000
patches: 0.0000
